In [3]:
!pip install -q transformers openpyxl

import json
import os
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, precision_score, recall_score, f1_score
import openpyxl
import zipfile

In [ ]:
BASE_PATH = "/kaggle/input/datasets/thaiphonghuan/dataset-training/dataset_structuring"
TRAIN_PATH = os.path.join(BASE_PATH, "train")
VAL_PATH = os.path.join(BASE_PATH, "val")
TEST_PATH = os.path.join(BASE_PATH, "test")

CV_VIEWS = ['skills', 'education', 'year_experience', 'experience', 'responsibilities',
            'projects', 'publications', 'achievement', 'certification', 'employment_history']
JD_VIEWS = ['skills', 'education', 'year_experience_requirement', 'experience_requirement', 'responsibilities']

# === CROSS-ENCODER CONFIG ===
ENCODER_NAME = "bert-base-multilingual-cased"
# Đổi tay 1 trong 4 model rồi chạy lại cả notebook:
#   "BAAI/bge-reranker-v2-m3"
#   "jinaai/jina-reranker-v2-base-multilingual"
#   "xlm-roberta-base"
#   "bert-base-multilingual-cased"
ENCODER_SLUG_MAP = {
    "BAAI/bge-reranker-v2-m3": "bge_reranker_v2_m3",
    "jinaai/jina-reranker-v2-base-multilingual": "jina_reranker_v2",
    "xlm-roberta-base": "xlmr_base",
    "bert-base-multilingual-cased": "mbert_cased",
}
ENCODER_SLUG = ENCODER_SLUG_MAP[ENCODER_NAME]
FUSION_NAME = "baseline_mlp"
CACHE_DIR = "/kaggle/working/cache"
CKPT_DIR = "/kaggle/working/checkpoints"
EXCEL_PATH = "/kaggle/working/results_crossencoder.xlsx"
MAX_LEN = 256

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Load cross-encoder backbone (FROZEN, no gradient)
tokenizer = AutoTokenizer.from_pretrained(ENCODER_NAME, trust_remote_code=True)
encoder = AutoModel.from_pretrained(ENCODER_NAME, trust_remote_code=True).to(device).eval()
for p in encoder.parameters():
    p.requires_grad_(False)
HIDDEN_DIM = encoder.config.hidden_size
print(f"Encoder: {ENCODER_NAME} | hidden_dim={HIDDEN_DIM} | FUSION={FUSION_NAME}")

# Cross-encoder forward + Masking matrix
Tokenize `[CLS] CV_view_i [SEP] JD_view_j [SEP]` cho 50 cặp, lấy CLS embedding (frozen). Cache `.npz` để tái sử dụng.

In [ ]:
def process_view(view_data):
    """Xử lý missing_section thành special token [EMPTY_SECTION]; trả về (text, exists_flag)."""
    if view_data is None: return "[EMPTY_SECTION]", 0
    if isinstance(view_data, list):
        clean_list = [str(x) for x in view_data if str(x).lower() != "missing_section"]
        if not clean_list:
            return "[EMPTY_SECTION]", 0
        return ". ".join(clean_list), 1
    val = str(view_data).lower()
    if val in ("missing_section", "", "none"):
        return "[EMPTY_SECTION]", 0
    return str(view_data), 1

def prepare_data_crossencoder(path, split_name):
    """
    Pre-compute CLS embeddings cho 50 (cv_view, jd_view) pairs mỗi sample.
    Input cross-encoder: tokenizer(cv_text, jd_text) -> "[CLS] cv [SEP] jd [SEP]".
    Output: cls [N, 50, H], mask [N, 50], y_reg [N], y_bin [N].
    """
    cache_path = f"{CACHE_DIR}/{ENCODER_SLUG}_{split_name}.npz"
    if os.path.exists(cache_path):
        d = np.load(cache_path)
        print(f"Loaded cache: {cache_path} | cls.shape={d['cls'].shape}")
        return d['cls'], d['mask'], d['y_reg'], d['y_bin']

    matched_files = glob.glob(os.path.join(path, "matched/*.json"))
    mismatch_files = glob.glob(os.path.join(path, "mismatch/*.json"))
    configs = [(matched_files, 1), (mismatch_files, 0)]

    all_cls, all_mask, all_yreg, all_ybin = [], [], [], []
    max_len = min(MAX_LEN, tokenizer.model_max_length)

    print(f"Cross-encoder forward (frozen): {path}")
    for files, b_label in configs:
        for fp in tqdm(files, desc=f"Folder {b_label}"):
            with open(fp, 'r', encoding='utf-8') as f:
                data = json.load(f)

            cv_struct = data['resume_structured']
            jd_struct = data['job_description_structured']
            cv_texts, cv_exists = zip(*[process_view(cv_struct.get(v)) for v in CV_VIEWS])
            jd_texts, jd_exists = zip(*[process_view(jd_struct.get(v)) for v in JD_VIEWS])

            # Build 50 pairs in row-major order: (cv_0,jd_0), (cv_0,jd_1)...(cv_9,jd_4)
            cv_batch = [cv_texts[i] for i in range(10) for j in range(5)]
            jd_batch = [jd_texts[j] for i in range(10) for j in range(5)]
            tok = tokenizer(cv_batch, jd_batch, padding=True, truncation=True,
                            max_length=max_len, return_tensors='pt').to(device)
            with torch.no_grad():
                out = encoder(**tok)
                cls = out.last_hidden_state[:, 0, :].cpu().numpy()  # [50, H]

            all_cls.append(cls)
            all_mask.append(np.outer(np.array(cv_exists), np.array(jd_exists)).flatten().astype(np.float32))
            all_yreg.append(data['target_label'] / 10.0)
            all_ybin.append(b_label)

    cls_arr = np.stack(all_cls).astype(np.float32)     # [N, 50, H]
    mask_arr = np.stack(all_mask).astype(np.float32)   # [N, 50]
    yreg = np.array(all_yreg, dtype=np.float32)
    ybin = np.array(all_ybin, dtype=np.float32)
    np.savez_compressed(cache_path, cls=cls_arr, mask=mask_arr, y_reg=yreg, y_bin=ybin)
    print(f"Saved cache: {cache_path} | cls.shape={cls_arr.shape}")
    return cls_arr, mask_arr, yreg, ybin

X_train, M_train, y_train, b_train = prepare_data_crossencoder(TRAIN_PATH, "train")
X_val,   M_val,   y_val,   b_val   = prepare_data_crossencoder(VAL_PATH,   "val")
X_test,  M_test,  y_test,  b_test  = prepare_data_crossencoder(TEST_PATH,  "test")
print(f"Train cls={X_train.shape}, mask={M_train.shape} | Val cls={X_val.shape} | Test cls={X_test.shape}")

# Baseline MLP với Trainable Scoring Head
- `ViewPairScorer`: Linear(H -> 1) + Sigmoid  -> alignment score A_ij in [0,1]
- `AdvancedFusionMLP` (giữ NGUYÊN): 50 -> 512 -> 128 -> 64 -> 1
- `FusionWithScorer`: ráp 2 module trên

In [ ]:
class CrossEncDataset(Dataset):
    def __init__(self, cls, mask, y_reg, y_bin):
        self.cls = torch.tensor(cls, dtype=torch.float32)
        self.mask = torch.tensor(mask, dtype=torch.float32)
        self.y_reg = torch.tensor(y_reg, dtype=torch.float32).view(-1, 1)
        self.y_bin = torch.tensor(y_bin, dtype=torch.float32).view(-1, 1)
    def __len__(self): return len(self.cls)
    def __getitem__(self, idx):
        return self.cls[idx], self.mask[idx], self.y_reg[idx], self.y_bin[idx]

class ViewPairScorer(nn.Module):
    """Mã hóa CLS thành điểm tương tác [0, 1]. Khởi tạo khắt khe giống bản Attention."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )
        nn.init.constant_(self.fc[2].bias, -2.0) 

    def forward(self, cls):
        return torch.sigmoid(self.fc(cls)).squeeze(-1)

class AdvancedFusionMLP(nn.Module):
    """MLP nhận 100 đầu vào (50 scores + 50 masks) để phân biệt dữ liệu thiếu."""
    def __init__(self, input_dim=100):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    def forward(self, a_ij, mask):
        # Ghép cặp (concatenation) tương tự cách bản Synced WeightedSum/Attention làm
        x = torch.cat([a_ij, mask], dim=-1) 
        return self.net(x)

class FusionWithScorer(nn.Module):
    """Wrapper tổng hợp. Trả về: (Final Score, Masked Alignment, Raw Alignment)."""
    def __init__(self, hidden_dim, fusion_module):
        super().__init__()
        self.scorer = ViewPairScorer(hidden_dim)
        self.fusion = fusion_module
        
    def forward(self, cls, mask):
        raw_a_ij = self.scorer(cls) 
        masked_a_ij = raw_a_ij * mask 
        final_score = self.fusion(masked_a_ij, mask)               
        return final_score, masked_a_ij, raw_a_ij                                 

# Loaders
train_loader = DataLoader(CrossEncDataset(X_train, M_train, y_train, b_train), batch_size=32, shuffle=True)
val_loader   = DataLoader(CrossEncDataset(X_val,   M_val,   y_val,   b_val),   batch_size=32)
test_loader  = DataLoader(CrossEncDataset(X_test,  M_test,  y_test,  b_test),  batch_size=32)

# Khởi tạo mô hình MLP với 100 input
model = FusionWithScorer(HIDDEN_DIM, AdvancedFusionMLP(input_dim=100)).to(device)

# --- HYPERPARAMETERS ĐỒNG BỘ TUYỆT ĐỐI ---
criterion_reg = nn.MSELoss() 
criterion_bin = nn.BCELoss()
optimizer = optim.AdamW(model.parameters(), lr=0.0003, weight_decay=0.05) 
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5)

print(f"BASELINE MLP (Synced) | Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Training

In [ ]:
# =========================
# Training config
# =========================
epochs = 150
best_val_loss = float('inf')
CKPT_PATH = f"{CKPT_DIR}/{FUSION_NAME}_{ENCODER_SLUG}.pth"

# Hệ số phạt đồng bộ với bản Attention tốt nhất
LAMBDA_PENALTY = 2.0 

# =========================
# History for plotting
# =========================
train_losses = []
val_losses = []
lrs = []

print(f"Bắt đầu Training Weighted Sum | Sync Mode | encoder={ENCODER_NAME}")

for epoch in range(epochs):
    # =========================
    # Train
    # =========================
    model.train()
    total_train_loss = 0.0
    
    for cls, mask, by_reg, by_bin in train_loader:
        cls = cls.to(device)
        mask = mask.to(device)
        by_reg = by_reg.to(device)
        by_bin = by_bin.to(device)

        optimizer.zero_grad()
        
        # Unpack 3 đầu ra
        pred, masked_a_ij, raw_a_ij = model(cls, mask)
        
        # 1. Task Loss: MSE + 0.2 * BCE
        task_loss = criterion_reg(pred, by_reg) + 0.2 * criterion_bin(pred, by_bin)
        
        # 2. Penalty Loss: phạt attention ở ô trống
        empty_penalty = torch.mean(((1 - mask) * raw_a_ij) ** 2)
        
        # 3. Activity Loss: thưởng ô có dữ liệu
        activity_loss = torch.mean(mask * (0.1 - masked_a_ij).clamp(min=0)) 
        
        # Tổng hợp loss
        loss = task_loss + (LAMBDA_PENALTY * empty_penalty) + (0.5 * activity_loss)
        
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        total_train_loss += loss.item()

    avg_train = total_train_loss / len(train_loader)

    # =========================
    # Validation
    # =========================
    model.eval()
    total_val_loss = 0.0

    with torch.no_grad():
        for cls, mask, by_reg, by_bin in val_loader:
            cls = cls.to(device)
            mask = mask.to(device)
            by_reg = by_reg.to(device)
            by_bin = by_bin.to(device)

            out, m_aij, r_aij = model(cls, mask)
            
            v_task_loss = criterion_reg(out, by_reg) + 0.2 * criterion_bin(out, by_bin)
            v_penalty = torch.mean(((1 - mask) * r_aij) ** 2)
            
            total_val_loss += (v_task_loss + (LAMBDA_PENALTY * v_penalty)).item()

    avg_val = total_val_loss / len(val_loader)

    # =========================
    # Scheduler + History
    # =========================
    scheduler.step(avg_val)

    train_losses.append(avg_train)
    val_losses.append(avg_val)
    lrs.append(optimizer.param_groups[0]['lr'])

    # =========================
    # Logging
    # =========================
    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch+1:3d} | "
            f"Train: {avg_train:.4f} | "
            f"Val: {avg_val:.4f} | "
            f"LR: {optimizer.param_groups[0]['lr']:.6f}"
        )

    # =========================
    # Checkpoint
    # =========================
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save({
            'scorer': model.scorer.state_dict(),
            'fusion': model.fusion.state_dict(),
            'hidden_dim': HIDDEN_DIM,
            'encoder': ENCODER_NAME,
            'epoch': epoch + 1,
            'val_loss': avg_val
        }, CKPT_PATH)

print(f"✅ Hoàn tất Training! Best Val Loss: {best_val_loss:.6f}")

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np

# =========================
# Safety check
# =========================
assert 'train_losses' in globals(), "❌ Chưa có train_losses"
assert 'val_losses' in globals(), "❌ Chưa có val_losses"

# =========================
# Config lưu file
# =========================
SAVE_DIR = "plots"
os.makedirs(SAVE_DIR, exist_ok=True)

SAVE_PATH = os.path.join(
    SAVE_DIR,
    "training_validation_loss.png"
)

# =========================
# Data
# =========================
epochs = len(train_losses)
x = np.arange(1, epochs + 1)

final_train = train_losses[-1]
final_val = val_losses[-1]

# =========================
# Plot
# =========================
plt.figure(figsize=(8, 5))
plt.plot(
    x,
    train_losses,
    label=f"Train Loss (final = {final_train:.4f})"
)
plt.plot(
    x,
    val_losses,
    label=f"Validation Loss (final = {final_val:.4f})"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)

# =========================
# Save figure
# =========================
plt.tight_layout()
plt.savefig(SAVE_PATH, dpi=300, bbox_inches="tight")
plt.close()

print(f"✅ Đã lưu plot tại: {SAVE_PATH}")

# Evaluation

In [ ]:
import torch
import numpy as np
import os
import json
import openpyxl
import zipfile
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score, precision_score, recall_score, f1_score

# --- 1. ĐỊNH NGHĨA HÀM LƯU EXCEL (Để tránh lỗi NameError) ---
def upsert_excel(path, sheet, encoder_name, metrics):
    sheets = ["weighted_sum", "baseline_mlp", "residual_mlp", "attention"]
    header = ["Encoder", "MAE", "MSE", "Accuracy", "Precision", "Recall", "F1-Score"]
    
    if not os.path.exists(path):
        wb = openpyxl.Workbook()
        wb.remove(wb.active)
        for s in sheets:
            ws = wb.create_sheet(s)
            ws.append(header)
    else:
        wb = openpyxl.load_workbook(path)
        for s in sheets:
            if s not in wb.sheetnames:
                ws = wb.create_sheet(s)
                ws.append(header)
                
    ws = wb[sheet]
    row_idx = None
    for r in range(2, ws.max_row + 1):
        if ws.cell(r, 1).value == encoder_name:
            row_idx = r
            break
    if row_idx is None:
        row_idx = ws.max_row + 1
        
    ws.cell(row_idx, 1).value = encoder_name
    for col, val in enumerate(metrics, start=2):
        ws.cell(row_idx, col).value = float(val)
    
    wb.save(path)
    print(f"✅ Excel updated: {path} | sheet='{sheet}' | row={row_idx}")

# --- 2. THỰC THI ĐÁNH GIÁ ---
ckpt = torch.load(CKPT_PATH)
model.scorer.load_state_dict(ckpt['scorer'])
model.fusion.load_state_dict(ckpt['fusion'])
model.eval()

all_preds, all_true_reg, all_true_bin = [], [], []
print(f"🚀 Đang đánh giá tập Test với model: {ENCODER_NAME}...")

with torch.no_grad():
    for cls, mask, by_reg, by_bin in test_loader:
        cls, mask = cls.to(device), mask.to(device)
        
        # Unpack 3 giá trị theo kiến trúc mới: (final_score, masked_aij, raw_aij)
        outputs, _, _ = model(cls, mask) 
        
        all_preds.extend(outputs.cpu().numpy().flatten())
        all_true_reg.extend(by_reg.numpy().flatten())
        all_true_bin.extend(by_bin.numpy().flatten())

all_preds = np.array(all_preds)
all_true_reg = np.array(all_true_reg)
all_true_bin = np.array(all_true_bin)
threshold = 0.5
all_preds_bin = (all_preds >= threshold).astype(int)

# Tính toán các chỉ số
mae = mean_absolute_error(all_true_reg, all_preds)
mse = mean_squared_error(all_true_reg, all_preds)
acc = accuracy_score(all_true_bin, all_preds_bin)
pre = precision_score(all_true_bin, all_preds_bin)
rec = recall_score(all_true_bin, all_preds_bin)
f1  = f1_score(all_true_bin, all_preds_bin)

print("\n" + "="*55)
print(f"Encoder: {ENCODER_NAME}")
print(f"Fusion : {FUSION_NAME}")
print("-" * 55)
print(f"{'MAE':<25} | {mae:.4f}")
print(f"{'MSE':<25} | {mse:.4f}")
print(f"{'Accuracy':<25} | {acc*100:.2f}%")
print(f"{'Precision (Match)':<25} | {pre:.4f}")
print(f"{'Recall (Match)':<25} | {rec:.4f}")
print(f"{'F1-Score':<25} | {f1:.4f}")
print("="*55)

print("\n🔍 Đối chiếu 10 mẫu trên tập Test:")
print(f"{'Thực tế':<10} | {'Điểm gốc':<10} | {'Dự đoán':<10} | {'Kết luận'}")
print("-" * 55)
for i in range(min(10, len(all_preds))):
    folder = "Matched" if all_true_bin[i] == 1 else "Mismatch"
    label  = "Match" if all_preds[i] >= threshold else "Mismatch"
    print(f"{folder:<10} | {all_true_reg[i]:<10.2f} | {all_preds[i]:<10.2f} | {label}")

# 3. LƯU KẾT QUẢ
upsert_excel(EXCEL_PATH, FUSION_NAME, ENCODER_NAME, [mae, mse, acc, pre, rec, f1])

zip_name = f"/kaggle/working/{FUSION_NAME}_{ENCODER_SLUG}.zip"
with zipfile.ZipFile(zip_name, "w") as z:
    z.write(CKPT_PATH, arcname=os.path.basename(CKPT_PATH))
print(f"📦 Saved zip: {zip_name}")